# 02 — Atomic release protocol, concurrency, and fault injection

This notebook executes the actual prototype release sequence:
execute and buffer a DuckDB answer, read an immutable constraint
snapshot, repair an external proposal, independently verify it, issue
a MAC-bound token, and atomically compare the current epoch/digests
before releasing the answer.

The atomicity evidence is an in-process Python service prototype around
DuckDB—not a claim of kernel-level integration or distributed
transactional correctness.


In [1]:
import subprocess
import sys

# Install only dependencies that are absent or version-sensitive in Colab.
# The final environment manifest records every resolved version.
packages = [
    "duckdb==1.3.2",
    "osqp==1.0.4",
    "sqlglot==27.0.0",
    "pyarrow==19.0.1",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", *packages]
)
print("Required packages are available.")


Required packages are available.


In [2]:
from pathlib import Path
import hashlib
import importlib
import json
import os
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_GOOGLE_DRIVE = True
if IN_COLAB and USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    default_root = "/content/drive/MyDrive/Arbiter_ICDE_2027"
else:
    default_root = "/content/Arbiter_ICDE_2027"

PROJECT_ROOT = Path(os.environ.get("ARBITER_PROJECT_ROOT", default_root))
active_path = PROJECT_ROOT / "active_run.json"
if not active_path.exists():
    raise FileNotFoundError(
        "Run Notebook 00 first. It creates active_run.json and the shared code."
    )
active = json.loads(active_path.read_text(encoding="utf-8"))
RUN_MODE = active["run_mode"]
if active.get("suite_version") != "2026-08-01-audited-v4":
    raise RuntimeError(
        "The active run was created by an older suite. Run the audited "
        "Notebook 00 before continuing."
    )
RUN_DIR = PROJECT_ROOT / active["run_directory"]
CODE_DIR = PROJECT_ROOT / "code"
if not CODE_DIR.exists():
    raise FileNotFoundError("Shared code is missing. Rerun Notebook 00.")
manifest_path = CODE_DIR / "code_manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError("Shared-code manifest is missing. Rerun Notebook 00.")
expected_hashes = json.loads(manifest_path.read_text(encoding="utf-8"))
for name, expected_hash in expected_hashes.items():
    module_path = CODE_DIR / name
    if not module_path.is_file():
        raise FileNotFoundError(f"Shared module is missing: {name}")
    observed_hash = hashlib.sha256(module_path.read_bytes()).hexdigest()
    if observed_hash != expected_hash:
        raise RuntimeError(f"Shared module hash mismatch: {name}")
code_dir_text = str(CODE_DIR.resolve())
if code_dir_text not in sys.path:
    sys.path.insert(0, code_dir_text)
for module_name in [
    "arbiter_core",
    "arbiter_workload",
    "arbiter_experiments",
    "arbiter_data",
]:
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
config = json.loads((RUN_DIR / "run_config.json").read_text(encoding="utf-8"))
print(f"Active run: {RUN_MODE}")
print(f"Run directory: {RUN_DIR}")


Mounted at /content/drive
Active run: full
Run directory: /content/drive/MyDrive/Arbiter_ICDE_2027/runs/full_audited_v4


In [3]:
import pandas as pd
import numpy as np
import duckdb
import shutil
import time

from arbiter_core import (
    AnswerBuffer,
    SnapshotStore,
    build_snapshot,
    exact_box_row_attacks,
    independent_verify,
    run_enforcement_method,
)
from arbiter_experiments import (
    run_fault_injection_campaign,
    run_stale_epoch_concurrency_campaign,
)

workload_dir = RUN_DIR / "00_workload"
proposal_dir = RUN_DIR / "01_proposals_attacks"
output_dir = RUN_DIR / "02_release_protocol"
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

all_assets = pd.read_csv(workload_dir / "sql_assets.csv")
assets = all_assets[
    all_assets["expected_supported"]
    .astype(str)
    .str.lower()
    .eq("true")
].copy()
relations = pd.read_csv(workload_dir / "compiled_relations.csv")
policy = pd.read_csv(proposal_dir / "controlled_policy.csv")
database_path = (
    PROJECT_ROOT
    / "state"
    / f"arbiter_sf{config['tpch_scale_factor']:g}.duckdb"
)
connection = duckdb.connect(str(database_path))


In [4]:
fault_frames = []
concurrency_frames = []
snapshot_bundle = {}
for dataset, dataset_assets in assets.groupby("dataset", sort=False):
    ordered_ids = dataset_assets["asset_id"].astype(str).tolist()
    dataset_relations = relations[relations["dataset"] == dataset]
    snapshot = build_snapshot(
        ordered_ids, dataset_relations, epoch=1
    )
    price_map = (
        policy[policy["dataset"] == dataset]
        .set_index("asset_id")["target_price"]
        .to_dict()
    )
    feasible = np.asarray(
        [price_map[item] for item in ordered_ids], dtype=float
    )
    attack = exact_box_row_attacks(
        snapshot,
        feasible,
        budget_fraction=0.50,
        limit=1,
        tolerance=config["tolerance"],
    )
    if not attack:
        raise RuntimeError(f"No violating test vector for {dataset}")
    violating = attack[0]["candidate"]
    faults = run_fault_injection_campaign(
        snapshot=snapshot,
        feasible_prices=feasible,
        violating_prices=violating,
        tolerance=config["tolerance"],
    )
    faults.insert(0, "dataset", dataset)
    fault_frames.append(faults)

    concurrency = run_stale_epoch_concurrency_campaign(
        snapshot=snapshot,
        feasible_prices=feasible,
        trials=config["concurrency_trials"],
        tolerance=config["tolerance"],
    )
    concurrency.insert(0, "dataset", dataset)
    concurrency_frames.append(concurrency)
    snapshot_bundle[dataset] = (
        dataset_assets,
        snapshot,
        feasible,
        violating,
    )

fault_results = pd.concat(fault_frames, ignore_index=True)
concurrency_results = pd.concat(
    concurrency_frames, ignore_index=True
)
fault_results.to_csv(
    output_dir / "fault_injection.csv", index=False
)
concurrency_results.to_csv(
    output_dir / "stale_epoch_concurrency.csv", index=False
)
display(
    fault_results.groupby(["case"], as_index=False)
    .agg(trials=("passed", "size"), pass_rate=("passed", "mean"))
)
display(
    concurrency_results.groupby(
        ["dataset", "schedule"], as_index=False
    )
    .agg(
        trials=("passed", "size"),
        pass_rate=("passed", "mean"),
        releases=("released", "sum"),
    )
)


,case,trials,pass_rate
0,asset_digest_tamper,3,1.0
1,asset_order_reversal,3,1.0
2,constraint_digest_tamper,3,1.0
3,fallback_failure,3,1.0
4,infinite_tolerance,3,1.0
5,infinity_input,3,1.0
6,mismatched_large_tolerance,3,1.0
7,missing_constraint_rows,3,1.0
8,missing_price,3,1.0
9,nan_input,3,1.0


,dataset,schedule,trials,pass_rate,releases
0,NYC Taxi 2024-01,simultaneous_race,100,1.0,18
1,NYC Taxi 2024-01,update_before_commit,100,1.0,0
2,TPC-H SF1,simultaneous_race,100,1.0,15
3,TPC-H SF1,update_before_commit,100,1.0,0
4,UCI Adult,simultaneous_race,100,1.0,12
5,UCI Adult,update_before_commit,100,1.0,0


In [5]:
# End-to-end buffered query release and forced stale-state rejection.
end_to_end_rows = []
for dataset, (
    dataset_assets,
    snapshot,
    feasible,
    violating,
) in snapshot_bundle.items():
    # Construct the next immutable state outside request timing;
    # only the atomic state transition belongs to the commit path.
    next_snapshot = build_snapshot(
        snapshot.asset_ids,
        relations[relations["dataset"] == dataset],
        epoch=snapshot.epoch + 1,
    )
    aggregate_representative = dataset_assets[
        dataset_assets["family"].str.contains(
            "aggregate_partition", na=False
        )
    ].iloc[0]
    rowset_representative = (
        dataset_assets.loc[
            ~dataset_assets["family"].str.contains(
                "aggregate_partition", na=False
            )
        ]
        .sort_values(["estimated_rows", "asset_id"])
        .iloc[0]
    )
    representatives = [
        ("aggregate_scalar", aggregate_representative),
        ("rowset_low_estimate", rowset_representative),
    ]
    for query_profile, representative in representatives:
        for scenario in ["normal", "stale_before_commit"]:
            repetitions = (
                (3 if config["run_mode"] == "pilot" else 25)
                if query_profile == "aggregate_scalar"
                else (1 if config["run_mode"] == "pilot" else 3)
            )
            for repetition in range(repetitions):
                store = SnapshotStore(
                    snapshot,
                    secret=b"arbiter-e2e-secret".ljust(32, b"!"),
                    verifier_tolerance=config["tolerance"],
                )
                query_start = time.perf_counter_ns()
                answer = connection.execute(
                    representative["sql"]
                ).fetchall()
                query_ms = (
                    time.perf_counter_ns() - query_start
                ) / 1e6
                buffer = AnswerBuffer(answer)
                current = store.read()
                gate_start = time.perf_counter_ns()
                repair = run_enforcement_method(
                    current,
                    violating,
                    method="arbiter_active",
                    tolerance=config["tolerance"],
                    max_sweeps=config["repair_sweeps"],
                    active_row_budget=config[
                        "repair_active_row_budget"
                    ],
                    step=config["repair_step"],
                    osqp_max_iter=config["osqp_max_iter"],
                )
                token = (
                    None
                    if repair.rejected
                    else store.issue_token(
                        current,
                        repair.prices,
                        tolerance=config["tolerance"],
                    )
                )
                if scenario == "stale_before_commit":
                    store.advance(next_snapshot)
                if token is None:
                    commit_released = False
                    commit_reason = repair.verification.reason
                    buffer.retain(commit_reason)
                else:
                    commit = store.compare_and_commit(
                        token, repair.prices, buffer
                    )
                    commit_released = commit.released
                    commit_reason = commit.reason
                gate_and_commit_ms = (
                    time.perf_counter_ns() - gate_start
                ) / 1e6
                end_to_end_rows.append(
                    {
                        "dataset": dataset,
                        "query_profile": query_profile,
                        "asset_id": representative["asset_id"],
                        "scenario": scenario,
                        "repetition": repetition,
                        "query_ms": query_ms,
                        "gate_and_commit_ms": gate_and_commit_ms,
                        "end_to_end_ms": query_ms + gate_and_commit_ms,
                        "answer_rows_buffered": len(answer),
                        "released": commit_released,
                        "buffer_released": buffer.released,
                        "reason": commit_reason,
                        "unsafe_release": bool(
                            commit_released
                            and not repair.verification.passed
                        ),
                    }
                )

end_to_end = pd.DataFrame(end_to_end_rows)
end_to_end.to_csv(
    output_dir / "end_to_end_release.csv", index=False
)
display(
    end_to_end.groupby(
        ["dataset", "query_profile", "scenario"],
        as_index=False,
    )
    .agg(
        trials=("released", "size"),
        released_rate=("released", "mean"),
        query_p50_ms=("query_ms", "median"),
        control_p50_ms=("gate_and_commit_ms", "median"),
        end_to_end_p50_ms=("end_to_end_ms", "median"),
        unsafe_releases=("unsafe_release", "sum"),
    )
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,dataset,query_profile,scenario,trials,released_rate,query_p50_ms,control_p50_ms,end_to_end_p50_ms,unsafe_releases
0,NYC Taxi 2024-01,aggregate_scalar,normal,25,1.0,38.171521,6.966538,45.898829,0
1,NYC Taxi 2024-01,aggregate_scalar,stale_before_commit,25,0.0,52.335772,118.739723,173.620113,0
2,NYC Taxi 2024-01,rowset_low_estimate,normal,3,1.0,326.041745,6.850100,332.806527,0
3,NYC Taxi 2024-01,rowset_low_estimate,stale_before_commit,3,0.0,299.634728,121.219337,422.858231,0
4,TPC-H SF1,aggregate_scalar,normal,25,1.0,1.967305,6.769266,8.815228,0
5,TPC-H SF1,aggregate_scalar,stale_before_commit,25,0.0,2.124811,187.037367,190.005459,0
6,TPC-H SF1,rowset_low_estimate,normal,3,1.0,43.288552,7.379556,50.708628,0
7,TPC-H SF1,rowset_low_estimate,stale_before_commit,3,0.0,38.692787,109.606824,150.702900,0
8,UCI Adult,aggregate_scalar,normal,25,1.0,1.614234,7.047933,8.738011,0
9,UCI Adult,aggregate_scalar,stale_before_commit,25,0.0,1.636763,192.055124,193.839507,0


In [6]:
quality = pd.DataFrame(
    [
        {
            "check": "all injected faults behave as specified",
            "passed": bool(fault_results["passed"].all()),
        },
        {
            "check": (
                "concurrent update/commit outcomes are linearizable "
                "and fail closed"
            ),
            "passed": bool(concurrency_results["passed"].all()),
        },
        {
            "check": "normal end-to-end requests release",
            "passed": bool(
                end_to_end.loc[
                    end_to_end["scenario"] == "normal", "released"
                ].all()
            ),
        },
        {
            "check": "end-to-end test covers scalar and row-set answers",
            "passed": bool(
                set(end_to_end["query_profile"])
                == {"aggregate_scalar", "rowset_low_estimate"}
                and end_to_end.loc[
                    end_to_end["query_profile"]
                    == "rowset_low_estimate",
                    "answer_rows_buffered",
                ].min()
                > 1
            ),
        },
        {
            "check": "stale end-to-end requests do not release",
            "passed": bool(
                ~end_to_end.loc[
                    end_to_end["scenario"]
                    == "stale_before_commit",
                    "released",
                ].any()
            ),
        },
        {
            "check": "no measured unsafe end-to-end release",
            "passed": bool(
                end_to_end["unsafe_release"].sum() == 0
            ),
        },
    ]
)
quality.to_csv(output_dir / "stage_quality_checks.csv", index=False)
display(quality)
if not quality["passed"].all():
    raise RuntimeError(
        "Notebook 02 failed. Do not claim an implemented atomic "
        "release protocol from this run."
    )
print("Notebook 02 PASS. Continue to Notebook 03.")


,check,passed
0,all injected faults behave as specified,True
1,concurrent update/commit outcomes are lineariz...,True
2,normal end-to-end requests release,True
3,end-to-end test covers scalar and row-set answers,True
4,stale end-to-end requests do not release,True
5,no measured unsafe end-to-end release,True


Notebook 02 PASS. Continue to Notebook 03.
